# `semsimula-diag` — a guided tour

Every diagnostic utility in the package, with a worked example of **how to
configure it**, **what it measures**, and **when you would reach for it**.

The notebook assumes you already have checkpoint files on disk and know
their paths — spikebatch bundles, prereload snapshots, or ordinary
checkpoints. Nothing here trains anything or mutates a run.

## How the package is organised

| module | what lives there |
|---|---|
| `semsimula_diag.replay` | finding and loading capture bundles; gradient isolation |
| `semsimula_diag.clipping` | per-group clipping, and both clip strategies |
| `semsimula_diag.probes.context` | `ProbeContext` — what replaces notebook globals |
| `semsimula_diag.probes.tokens` | token-degeneracy forensics (no model needed) |
| `semsimula_diag.probes.stiffness` | curvature / effective-rank probes (model needed) |
| `semsimula_diag.report` | `ProbeResult`, the shared result shape |

## What you need to run each section

Sections 1–8 need **only a bundle file** and run fine on CPU in seconds.
Section 9 needs **a built model** with an anisotropic Gaussian `V_theta`,
and is the slow one (tens of seconds per call on CPU).

---
## 0. Configuration — the paths you supply

Everything downstream is derived from these four values, so this is the only
cell you should need to edit.

**What the naming convention is.** Bundles are named
`{ckpt_prefix}_step{N}_{suffix}.pt`, e.g.
`fock_cfc_owt_..._baoab_cfc_step87196_spikebatch.pt`. `BundleStore` builds
that filename for you, so you pass step *numbers*, never paths.

**Why `ARCHIVE_ROOT` is separate.** The live checkpoint directory is a
rotating ring — a fixed number of the most recent captures, evicting the
oldest on every new one. A permanent archive sits alongside it in
`{ARCHIVE_ROOT}/spikebatch_archive/` and `{ARCHIVE_ROOT}/prereload_archive/`.
Give both and lookups fall back automatically; a bundle evicted from the ring
is still found.

In [ ]:
from pathlib import Path

# --- EDIT THESE ------------------------------------------------------------
CKPT_DIR     = Path.home() / 'Downloads'          # the live checkpoint dir
ARCHIVE_ROOT = Path.home() / 'Downloads'          # parent of *_archive/ dirs
CKPT_PREFIX  = ('fock_cfc_owt_xi5long_topk16_dt32da16_mh4_aniso_dcvt5x8'
                '_L8probe_ob_untied_wsd_e5c_plgate_rep0.05_fockreg0.005'
                '_g0.1_baoab_cfc')
SPIKE_STEP   = 87196      # a spikebatch capture (batch + RNG + weights)
# ---------------------------------------------------------------------------

DEVICE = 'cpu'            # 'cuda' if you have a GPU; only section 9 cares

import semsimula_diag
print('semsimula_diag', semsimula_diag.__version__)
print('ckpt dir exists :', CKPT_DIR.exists())

---
## 1. `BundleStore` — locating a capture without knowing its path

**What it does.** Turns a step number into a loaded bundle, looking in the
live directory first and then the permanent archive.

**Why you need it.** The live ring rotates on every new capture, whether or
not you have analysed the older ones yet — one 24h session produced 13
captures against a 12-deep ring. So the bundle you want is frequently *not*
where it was written. Hard-coding paths breaks silently the moment that
happens; `BundleStore` makes the fallback automatic and tells you when it
fires.

**Use it when** you have a step number from a training log (a `[spike]` line,
a watchdog reload) and want the bundle behind it.

In [ ]:
from semsimula_diag import BundleStore

store = BundleStore(
    ckpt_dir=CKPT_DIR,
    ckpt_prefix=CKPT_PREFIX,
    archive_root=ARCHIVE_ROOT,   # omit -> defaults to ckpt_dir.parent
    verbose=True,                # prints a line when the archive is used
)

# What filename will it look for?
print(store.filename(SPIKE_STEP, 'spikebatch'))

# Where did it actually find it? -> (path, 'live'|'archive') or (None, None)
path, source = store.resolve(SPIKE_STEP, 'spikebatch')
print('source:', source)
print('path  :', path)

### The three bundle kinds

`suffix` selects which artefact you want, and they are **not**
interchangeable:

| suffix | contains | replayable? |
|---|---|---|
| `spikebatch` | weights **+ the batch + RNG state** | **yes** — the only kind that is |
| `prereload` | weights only (saved before a watchdog reload) | no — no batch, no RNG |
| *(a regular checkpoint)* | weights + optimizer state | no |

**This distinction decides which probes you can run.** Anything that replays
a step needs `spikebatch`. Weight-only probes — the whole stiffness family —
work with any of the three.

In [ ]:
for suffix in ('spikebatch', 'prereload'):
    p, src = store.resolve(SPIKE_STEP, suffix)
    print(f'{suffix:12s} -> {src or "NOT FOUND"}')

# A missing bundle raises, naming BOTH directories it searched, so an
# evicted-and-never-archived capture is diagnosable rather than just absent.
try:
    store.load(999_999)
except FileNotFoundError as e:
    print('\nFileNotFoundError:\n ', str(e)[:200], '...')

---
## 2. What is actually inside a bundle

**Why look.** Two details here bite people, and both are cheap to check
before you write code against a bundle.

1. **`batches` holds numpy arrays, not tensors.** Feeding them straight to
   an embedding raises `argument 'indices' must be Tensor, not
   numpy.ndarray`. Convert with `torch.as_tensor(...).long()`.
2. **`rng_state_cpu` must stay a CPU `ByteTensor`.** `BundleStore.load`
   always uses `map_location='cpu'` for exactly this reason — moving RNG
   state to a device breaks `torch.set_rng_state`.

`top_groups` is the per-group gradient norm the *live* training step
recorded. It is your ground truth when checking a replay.

In [ ]:
import torch

bundle, path = store.load(SPIKE_STEP)

print('keys:', sorted(bundle))
print(f"\nstep               : {bundle['step']}")
print(f"pre_clip_grad_norm : {bundle['pre_clip_grad_norm']}")
print(f"grad_accum         : {bundle['grad_accum']}")
print(f"microbatches       : {len(bundle['batches'])}")

x0, y0 = bundle['batches'][0]
print(f"batch[0] type      : {type(x0).__name__}  shape {x0.shape}")
print(f"rng_state_cpu      : {bundle['rng_state_cpu'].dtype} on "
      f"{bundle['rng_state_cpu'].device}")

print('\ntop_groups recorded live at capture time:')
for k, v in sorted(bundle['top_groups'].items(), key=lambda kv: -kv[1]):
    print(f'  {k:36s} {v:10.2f}')

---
## 3. `GradClipConfig` — how a parameter name becomes a clip group

**What it does.** Maps every parameter to a *group* and a clip threshold, by
substring match on the parameter's name. Unmatched names fall back to their
top-level attribute name at `default_clip`.

**Why it exists as an object.** Every norm-measuring utility needs the same
grouping. Passing one config around means the watchdog, the optimizer step,
and any offline probe all agree on what "the `register` group" means.

### The ordering trap — worth internalising

`overrides` is scanned **in insertion order and the first substring hit
wins**. `creation_gate_qkv.log_tau` matches *both* `log_tau` and
`creation_gate`. If `creation_gate` came first, the `log_tau` entry would be
dead config that silently never fires — and you would not get an error, just
wrong groupings.

In [ ]:
from semsimula_diag import GradClipConfig, assign_clip_group

cfg = GradClipConfig(
    default_clip=1.0,
    overrides={
        'V_phi': 0.3,
        'log_tau': 0.3,            # MUST precede 'creation_gate'
        'creation_gate': 0.3,
        'destruction_gate': 0.3,
        'reverse_channel_scale': 0.1,
        'reverse_ch': 0.1,
        'register': 0.3,
        'depth_code': 0.25,
    },
    watchdog_exclude_groups=frozenset({'override:reverse_channel_scale',
                                       'override:reverse_ch'}),
    clip_then_sum_groups=frozenset({'E', 'P'}),
    clip_then_sum_threshold=0.3,
)

for name in ['creation_gate_qkv.log_tau', 'creation_gate_qkv.W_Q',
             'V_theta.depth_code', 'register_embed', 'E.weight', 'score_head']:
    group, thr = assign_clip_group(name, cfg)
    print(f'{name:28s} -> {group:28s} clip={thr}')

### `watchdog_exclude_groups` — a deliberate blind spot

Excluded groups are **still clipped normally**; they are just left out of the
*aggregate* norm. That keeps the watchdog from false-triggering on a group's
normal warmup ramp — but it also means the watchdog cannot see them at all.

**Practical consequence:** a capture can report a total of 355.5 while
`reverse_channel_scale` alone was 823.2. When you are reading a spike, always
look at the per-group numbers, not just the total.

---
## 4. Measuring and clipping per group

Two functions, and the difference matters:

- **`per_group_grad_norms`** is **read-only**. Use it to observe.
- **`clip_grads_per_group`** **mutates** `.grad` in place and returns the
  norms measured *before* it rescaled. Use it in a training step.

**Use `per_group_grad_norms` when** checking a replay against a bundle's
`top_groups`, or inspecting gradients without disturbing them.

In [ ]:
import torch.nn as nn
from semsimula_diag import per_group_grad_norms, clip_grads_per_group

def toy():
    m = nn.Module()
    m.E = nn.Parameter(torch.zeros(4))
    m.register_embed = nn.Parameter(torch.zeros(4))
    m.reverse_channel_scale = nn.Parameter(torch.zeros(1))
    with torch.no_grad():
        m.E.grad = torch.full((4,), 3.0)                    # norm 6.0
        m.register_embed.grad = torch.full((4,), 5.0)       # norm 10.0
        m.reverse_channel_scale.grad = torch.full((1,), 100.0)   # excluded
    return m

m = toy()
print('read-only:', {k: round(v, 2) for k, v in per_group_grad_norms(m, cfg).items()})
print('grads untouched:', float(m.E.grad.norm()))

m = toy()
total, per_group = clip_grads_per_group(m, cfg)
print('\npre-clip per group:', {k: round(v, 2) for k, v in per_group.items()})
print(f'aggregate (excludes reverse_channel_scale): {float(total):.2f}')
print(f'  ... note 100.0 is missing from it, though that grad WAS clipped: '
      f'{float(m.reverse_channel_scale.grad.norm()):.2f}')

---
## 5. `ClipThenSum` — clip order changes the answer

**What it does.** Clips each configured group's contribution **per
microbatch**, then sums — instead of summing across all microbatches and
clipping once at the end.

**Why it matters.** These are genuinely different gradients, not an
implementation detail. Two microbatches each of norm 6.0 in the same
direction, at this config's `clip_then_sum_threshold=0.3`:

- clip-then-sum → clip each to 0.3, sum → **0.6**
- sum-then-clip → sum to 12.0, clip → **0.3**

A factor of two here, and it grows with the number of microbatches.
The cell below prints exactly these numbers.

**Use it when** replaying a capture from a run that used clip-then-sum
live. If you skip it, the affected groups come back wildly inflated — raw
and unclipped across every microbatch — and your replay silently disagrees
with training.

It is a **true no-op** when `clip_then_sum_groups` is empty, so it is safe
to wrap around any accumulation loop unconditionally.

In [ ]:
from semsimula_diag import ClipThenSum

model_cts = nn.Module()
model_cts.E = nn.Parameter(torch.zeros(4))

cts = ClipThenSum(model_cts, cfg)
print('active:', cts.active, '| groups:', sorted(cts.params))

for _ in range(2):                                  # two microbatches
    with torch.no_grad():
        model_cts.E.grad = torch.full((4,), 3.0)    # norm 6.0 each
    cts.apply_microbatch()                          # clip to 0.3, fold, zero
cts.splice_back()                                   # install the running total

print(f'clip_then_sum result : {float(model_cts.E.grad.norm()):.3f}')

ref = nn.Module(); ref.E = nn.Parameter(torch.zeros(4))
with torch.no_grad():
    ref.E.grad = torch.full((4,), 6.0)              # the same two, summed
nn.utils.clip_grad_norm_([ref.E], 0.3)
print(f'sum_then_clip result : {float(ref.E.grad.norm()):.3f}')

---
## 6. `isolated_grads` — probing without corrupting a live step

**What it does.** Snapshots every `.grad`, runs your block, restores them —
**including on the exception path**.

**Why you need it.** A probe that leaves gradients behind silently poisons
the next optimizer step. This is the invariant that makes it safe to call a
diagnostic mid-training.

**Use it whenever** a probe calls `.backward()` on a model that is also being
trained.

In [ ]:
from semsimula_diag import isolated_grads, grad_snapshot, grad_restore

m = nn.Module()
m.w = nn.Parameter(torch.zeros(3))
with torch.no_grad():
    m.w.grad = torch.full((3,), 2.0)

try:
    with isolated_grads(m):
        with torch.no_grad():
            m.w.grad = torch.full((3,), 99.0)   # a probe scribbles
        raise RuntimeError('probe blew up')
except RuntimeError as e:
    print('probe raised:', e)

print('grad restored anyway:', m.w.grad.tolist())

# the lower-level pair, if you need manual control
saved = grad_snapshot(m); grad_restore(m, saved)

---
## 7. `ProbeContext` — what replaces the notebook globals

**What it does.** Bundles everything a probe might need — model, device,
bundle store, clip config, loss weights — into one object passed explicitly.

**Why it exists.** In the notebook, probes reach into module globals
(`model`, `DEVICE`, `_GRAD_CLIP_CFG`, `LAMBDA_V`, …). That is what makes the
diagnostics impossible to load without the training cell having run first.

**Missing configuration raises.** This is deliberate and hard-won: the
notebook reads some of its settings with `globals().get(...)`, which returns
`None` instead of raising — so an unset value silently degraded clip-then-sum
into sum-then-clip and produced a plausible-looking but wrong answer.
`require()` reports **every** missing field at once, with what each is for.

In [ ]:
from semsimula_diag import ProbeContext
from semsimula_diag.probes import MissingContextError

ctx = ProbeContext(
    model=nn.Linear(2, 2),     # replaced with the real model in section 9
    device=DEVICE,
    store=store,
    clip_cfg=cfg,
    lambda_v=1e-2, lambda_fock=5e-3, fock_eps=1e-6,
    register_repulsion=True,
)

ctx.require('model', 'store', 'clip_cfg')      # fine
print('required fields present')

try:
    ctx.require('forward_fn', 'batch_provider')
except MissingContextError as e:
    print('\nMissingContextError:\n ', e)

---
## 8. Token probes — what the model was *reading*

**What they do.** Pure CPU bookkeeping over a capture's token ids. No model,
no GPU, no weights touched — safe to call at any time.

**Two metrics, and you need both:**

- `max_repeat_run` — longest run of the *same token back-to-back*. Blind to
  phrase-level repetition.
- `unique_token_ratio` — distinct tokens / length. Catches templated text
  that `max_repeat_run` scores as perfectly clean.

**The two functions ask opposite questions:**

| function | question | when to use |
|---|---|---|
| `inspect_spike_tokens` | *search*: which rows in this capture look degenerate? | first look at a new capture |
| `decode_hot_rows` | *test*: rows already named by gradient attribution — where do they rank? | after attribution names a row |

`decode_hot_rows` is the honest direction. Searching for a degenerate row and
finding one proves little; taking the row the gradient actually implicated
and asking where it sits in its own batch is what falsified "degenerate text
causes these spikes".

In [ ]:
from semsimula_diag.probes import tokens

# The metric primitive, on hand-made input
print('back-to-back :', tokens.row_degeneracy([7, 7, 7, 1, 2, 3]))
print('templated    :', tokens.row_degeneracy([1, 2, 3, 4] * 4),
      '<- max_repeat_run=1, but uniq ratio exposes it')

# Search direction: rank every row in the capture
res = tokens.inspect_spike_tokens(ctx, SPIKE_STEP, microbatches=[0],
                                  verbose=False)
print(f'\n{res.probe_name}: {res.metrics}')
print('worst 3 rows by max_repeat_run:')
for row in res.raw['rows'][:3]:
    print(f"  mb{row['microbatch']} row{row['row']:3d}  "
          f"repeat={row['max_repeat_run']:3d}  "
          f"uniq={row['unique_token_ratio']:.3f}")

In [ ]:
# Test direction: rows named elsewhere, ranked against their own batch.
# Pass a real tokenizer (anything with .decode) to also see the text.
hot = tokens.decode_hot_rows(ctx, SPIKE_STEP, hot_rows=[(0, 0), (0, 3)],
                             tokenizer=None, verbose=True)
print('\nA high rank number = NOT degenerate relative to its batch.')

---
## 9. Stiffness probes — curvature, and the rank decision

**What they measure.** The geometry of the anisotropic Gaussian
$V_\theta$: how sharp its wells are, and how many directions each one
actually uses.

These need a **real model** whose `V_theta` exposes `context_components`
and `harmonic_terms`. They hook the forward pass to capture the *realised*
$B_k$ — the matrix the forward actually used, after any cap — then restore
the original method in a `finally`. **Weights and training mode are left
exactly as found.**

They compute **no gradients**, so they work from any checkpoint kind
(spikebatch, prereload, or ordinary).

> Supply your own model below. The cell is written so the rest of the
> notebook still runs if you skip it.

In [ ]:
# --- supply your model here -------------------------------------------------
# It must have .V_theta with context_components()/harmonic_terms(), and
# .cfg.dt / .compute_mass(x) for stiffness_report.
model = None        # e.g. build_model(); model.load_state_dict(...)
# ---------------------------------------------------------------------------

HAVE_MODEL = model is not None
if HAVE_MODEL:
    model.load_state_dict(bundle['model_state_dict'], strict=True)
    model.to(DEVICE)

    def batch_provider(n):
        xb, _ = bundle['batches'][0]          # a fixed, neutral batch
        return torch.as_tensor(xb[:n]).long()

    ctx = ProbeContext(model=model, device=DEVICE, store=store,
                       clip_cfg=cfg, batch_provider=batch_provider)
    print('model ready; ProbeContext rebuilt')
else:
    print('No model supplied - section 9 cells will skip.')

### 9a. `sigma_lr_spectrum_report` — is the rank budget being used?

**What it returns.** The full singular-value spectrum of each $B_k$, reduced
to a **participation ratio** $\mathrm{PR} = (\sum \sigma_i^2)^2 / \sum
\sigma_i^4$, which lives in $[1, r]$ — an effective rank.

**Read `fro_p50` first.** It is the Frobenius norm. If the cap is *not*
binding, the rank question is premature: rank only becomes a pure
redistribution knob once total magnitude is already pinned.

**Then read `pr_p50`** against rank $r$:

| `pr_p50` (of rank 4) | reading |
|---|---|
| ≥ 3.0 | budget saturated — a higher rank has a real case |
| ≤ 2.0 | budget unused — higher rank would be wasted parameters |
| 2.0–3.0 | ambiguous — weigh against the spike comparison in 9b |

**Use it when** deciding `ANISO_RANK` for a future run.

In [ ]:
from semsimula_diag.probes import stiffness

if HAVE_MODEL:
    x = ctx.neutral_batch(2)          # small: this is the slow probe
    res = stiffness.sigma_lr_spectrum_report(ctx, x)
    for k, v in res.metrics.items():
        print(f'  {k:18s} {v}')
    print(f'  spectrum (sigma_i/sigma_1): '
          f'{[round(v, 4) for v in res.raw["spectrum"]]}')

    pr, rank = res.metrics['pr_p50'], res.metrics['rank']
    print(f'\n  PR {pr:.2f} of max {rank}  ({100*pr/rank:.0f}% of the budget)')
    print(f'  Frobenius cap binding? fro_p50 = {res.metrics["fro_p50"]:.6f}')

### 9b. `spectrum_across_checkpoints` — is a spike a spectral collapse?

**What it does.** Runs 9a across several checkpoints on one **fixed** batch,
so the only thing varying is the weights.

**The question it answers.** Does the participation ratio *drop* at a spike
relative to the best checkpoint? A collapse onto one dominant direction is a
different failure mode from a uniformly over-sharp well, and they call for
different fixes.

**Note `include_prereload`.** Prereload snapshots are weights-only and cannot
be replayed — but this probe only needs weights, so they work here. That
makes them useful evidence you would otherwise have to discard.

In [ ]:
if HAVE_MODEL:
    reports = stiffness.spectrum_across_checkpoints(
        ctx,
        step_tags=(SPIKE_STEP,),      # spikebatch bundles
        include_prereload=(),         # e.g. (85885,) — weights-only snapshots
        n_batch=2,
    )
    print(f'{"checkpoint":34s} {"pr_p50":>8s} {"fro_p50":>9s}')
    for label, r in reports.items():
        print(f'{label:34s} {r.metrics["pr_p50"]:8.3f} '
              f'{r.metrics["fro_p50"]:9.5f}')

### 9c. `sigma_lr_report` and `bracket_precision_lr_max`

`sigma_lr_report` gives the raw $\sigma_{\max}(B_k)^2$ distribution — the
quantity a `precision_lr_max` budget caps **directly**. It is deliberately
*not* mixed with the bump weight or the diagonal precision, unlike
`stiffness_report`'s $\omega \Delta t$.

`bracket_precision_lr_max` runs it across a healthy checkpoint and several
spike checkpoints, so you can see the range a candidate budget has to sit
between.

**Use these when** choosing or sanity-checking `precision_lr_max`.

In [ ]:
if HAVE_MODEL:
    r = stiffness.sigma_lr_report(ctx, x)
    print('sigma_max(B_k)^2:',
          {k: round(v, 5) for k, v in r.metrics.items()})

    brackets = stiffness.bracket_precision_lr_max(ctx, step_tags=(SPIKE_STEP,),
                                                  n_batch=2)
    print()
    for label, rep in brackets.items():
        print(f'{label:34s} p99={rep.metrics["p99"]:.4f} '
              f'max={rep.metrics["max"]:.4f}')

### 9d. `stiffness_report` — the integrator stability wall

**What it measures.** The distribution of $\omega \Delta t$, where
$\omega = \sqrt{k/m}$. The explicit kick in the `baoab_cfc` integrator is
stable only for $\omega \Delta t < 2$; above ~1 it is marginal.

**Why it forces the integrator.** The wall is a property of that specific
kick, so the probe temporarily sets `baoab_cfc` regardless of how the model
is configured — and restores your setting afterwards.

**Use it when** you suspect the integrator itself, rather than the loss, is
what is blowing up — and read `frac_unstable` / `frac_marginal`, not just
the median.

In [ ]:
if HAVE_MODEL:
    r = stiffness.stiffness_report(ctx, x)
    for k, v in r.metrics.items():
        print(f'  {k:16s} {v:.6f}' if isinstance(v, float) else f'  {k:16s} {v}')
    print('\n  frac_unstable is the fraction above the omega*dt = 2 wall.')

---
## 10. `ProbeResult` — one shape for every probe

**What it is.** The common return type. Every probe fills `metrics`, and
optionally `per_layer`, `per_group`, and `raw` (the full original report).

**Why it matters.** One shape means one code path to render, diff, or
serialise any probe's output — and it makes results comparable across runs
instead of living in ad-hoc prints.

**`to_json` omits `raw` by default**, since `raw` can be large. JSON turns
int keys into strings, so `from_json` converts `per_layer` keys back.

In [ ]:
from semsimula_diag import ProbeResult

r = ProbeResult(
    probe_name='layer_profile', step_tag=SPIKE_STEP,
    fidelity_gap_pct=0.0002,
    metrics={'L0_hgrad': 0.30, 'L7_hgrad': 0.00},
    per_layer={0: 0.30, 7: 0.00},
    per_group={'override:register': 2169.18},
    raw={'big': list(range(1000))},
)

print('to_dict() keys      :', sorted(r.to_dict()))
print('raw included?       :', 'raw' in r.to_dict(),
      '| with include_raw:', 'raw' in r.to_dict(include_raw=True))

back = ProbeResult.from_json(r.to_json())
print('per_layer keys back :', list(back.per_layer), '(ints, not strings)')

# Archive a result next to a run's other diagnostic outputs
# Path('probe_87196.json').write_text(r.to_json(indent=2))

---
## What is not in the package yet

These still live only in the training notebook. All of them **replay** a
capture (forward + backward), which is why they need
`ProbeContext.forward_fn` — you pass the run's own `forward_with_vreg` so
the replay matches training exactly.

| notebook function | destined module |
|---|---|
| `replay_spike_batch` | `probes.layer_profile` |
| `attribute_spike_rows` | `probes.row_attribution` |
| `replay_precision_cap_ablation`, `replay_curvature_rebalance_ablation` | `probes.precision_cap` |
| `replay_clip_ablation` | `probes.clip_order` |
| `replay_integrator_ablation` | `probes.integrator` |
| `probe_gate_saturation`, `sweep_log_tau_history`, `probe_hot_rows` | `probes.tau_saturation` |

A practical note if you replay by hand in the meantime: a replay must
reproduce the live loop's clip-then-sum mechanics (section 5) or the affected
groups come back inflated by orders of magnitude — and nothing will warn you.

See `MIGRATION.md` for status and the reasoning behind the porting order.

---
---
# Part II — GPU verification of the replay probes

**Everything below needs a CUDA runtime.** Run it on the A100 session.

## Why this section exists

The probes in Part I are weight-only or batch-only: they were verified on
CPU against real bundles. The replay-based probes are different — they are
only meaningful if they reproduce the gradients training *actually saw*, and
that has **not** yet been confirmed for this package.

What is known:

- The training notebook's own `replay_spike_batch` reproduces step 87196 to
  **0.0002%** and step 90360 to **0.0003%** on an A100 — bit-exact.
- A CPU replay of the same bundles does **not** reproduce them, for reasons
  not yet root-caused.

So a green CPU test proves nothing about replay correctness. These cells
check the ported probes against the known-good A100 numbers and print a
pass/fail table.

## What you need first

Run the training notebook's Cells **0, 1, 1b, 1c, 2, 3, 4, 5, 6d** (skip
Cell 6), plus the shim cell that defines `GRAD_CLIP_OVERRIDES`,
`WATCHDOG_EXCLUDE_GROUPS`, `CLIP_THEN_SUM_GROUPS`, `PER_GROUP_CLIP`,
`CLIP_THEN_SUM_THRESHOLD`, `_GRAD_CLIP_CFG` and `forward_with_vreg`.

Then run this notebook's Part I config cell, and the cells below.

## V1. Build a `ProbeContext` wired to the live run

**What this does.** Connects the package to the notebook's own objects — the
real `model`, and critically the run's own `forward_with_vreg`, so the replay
uses training's exact loss composition rather than a reimplementation.

**The single most important field is `clip_cfg`.** If
`clip_then_sum_groups` / `clip_then_sum_threshold` do not match the live run,
the affected groups come back inflated by orders of magnitude and *nothing
warns you* — that is precisely the failure that cost a debugging round-trip
on the A100. The assertion below is cheap insurance.

In [ ]:
import torch
from semsimula_diag import BundleStore, GradClipConfig, ProbeContext

assert torch.cuda.is_available(), 'Part II needs a GPU runtime'
for _n in ('model', 'forward_with_vreg', 'GRAD_CLIP_OVERRIDES',
           'WATCHDOG_EXCLUDE_GROUPS', 'CLIP_THEN_SUM_GROUPS',
           'CLIP_THEN_SUM_THRESHOLD'):
    assert _n in globals(), f'{_n} not defined - run the notebook cells + shim first'

gpu_cfg = GradClipConfig(
    default_clip=GRAD_CLIP,
    overrides=GRAD_CLIP_OVERRIDES,
    watchdog_exclude_groups=frozenset(WATCHDOG_EXCLUDE_GROUPS),
    clip_then_sum_groups=frozenset(CLIP_THEN_SUM_GROUPS),
    clip_then_sum_threshold=CLIP_THEN_SUM_THRESHOLD,
)

gpu_store = BundleStore(ckpt_dir=CKPT_DIR, ckpt_prefix=CKPT_PREFIX,
                        archive_root=GDRIVE_ROOT, verbose=True)

gpu_ctx = ProbeContext(
    model=model, device='cuda', store=gpu_store, clip_cfg=gpu_cfg,
    lambda_v=LAMBDA_V, lambda_fock=LAMBDA_FOCK_REG, fock_eps=FOCK_REG_EPS,
    register_repulsion=REGISTER_REPULSION,
    # the run's OWN loss function, not a reimplementation
    forward_fn=lambda m, x, y, c: forward_with_vreg(
        x, y, c.lambda_v, c.lambda_fock, c.fock_eps),
)

# clip_then_sum must resolve to real parameters, or the replay silently
# degrades to sum_then_clip
from semsimula_diag import clip_then_sum_params
_cts = clip_then_sum_params(model, gpu_cfg)
assert _cts, 'clip_then_sum resolved to NOTHING - the replay would be wrong'
print('clip_then_sum groups resolved:', sorted(_cts))
print('ProbeContext ready on', gpu_ctx.device)

## V2. The gate: `layer_profile` fidelity

**This is the test that matters.** Everything else in Part II is only
trustworthy if this passes.

`fidelity_gap_pct` compares the replayed aggregate against what the live
training step recorded in the bundle. Reference values from the A100 run of
the notebook's own function:

| step | expected gap |
|---|---|
| 87196 | 0.0002% |
| 90360 | 0.0003% |

**Reading the result.** Under ~0.01% is a pass. A few percent means the
context does not match the live run — check `clip_cfg` first. Orders of
magnitude off means the replay is not reproducing the step at all, and no
attribution from any replay probe should be believed.

In [ ]:
from semsimula_diag.probes import layer_profile

VERIFY = {}      # collects pass/fail for the summary table

EXPECTED_GROUPS = {
    87196: {'override:register': 2169.18, 'override:depth_code': 927.84,
            'override:creation_gate': 902.07,
            'override:reverse_channel_scale': 549.68, 'V_theta': 252.61,
            'override:reverse_ch': 147.66, 'override:log_tau': 36.06,
            'raw_m_bias': 32.89},
    90360: {'override:reverse_channel_scale': 490.75,
            'override:depth_code': 405.74, 'override:creation_gate': 338.65,
            'V_theta': 134.56, 'override:V_phi': 122.93,
            'override:reverse_ch': 109.34, 'override:register': 84.86,
            'override:log_tau': 42.26},
}

profiles = {}
for tag in (87196, 90360):
    print(f'\n{"="*70}\nlayer_profile.replay_spike_batch({tag})\n{"="*70}')
    res = layer_profile.replay_spike_batch(gpu_ctx, tag, verbose=True)
    profiles[tag] = res
    ok = res.fidelity_gap_pct is not None and res.fidelity_gap_pct < 0.01
    VERIFY[f'layer_profile fidelity @{tag}'] = (
        ok, f'{res.fidelity_gap_pct:.4f}% (want < 0.01%)')
    print(f'\n  -> fidelity {res.fidelity_gap_pct:.4f}%  '
          f'{"PASS" if ok else "FAIL"}')

## V3. Per-group norms against the known-good values

**Why check these separately.** The fidelity gap is an aggregate — it could
in principle be right while individual groups are wrong. These are the exact
per-group numbers the A100 produced, so a mismatch localises the problem to a
specific group rather than leaving you with one summary number.

**A useful diagnostic if it fails:** if `E` and `P` appear near the top with
values in the hundreds or thousands, clip-then-sum is not being applied —
they should not appear in the top groups at all.

In [ ]:
for tag, expected in EXPECTED_GROUPS.items():
    got = profiles[tag].per_group
    print(f'\nstep {tag}:')
    print(f'  {"group":34s} {"expected":>10s} {"got":>10s} {"":>6s}')
    worst = 0.0
    for k, want in sorted(expected.items(), key=lambda kv: -kv[1]):
        have = got.get(k)
        if have is None:
            print(f'  {k:34s} {want:10.2f} {"MISSING":>10s}')
            worst = float("inf"); continue
        rel = abs(have - want) / max(want, 1e-9) * 100
        worst = max(worst, rel)
        print(f'  {k:34s} {want:10.2f} {have:10.2f} {rel:5.2f}%')
    ok = worst < 0.1
    VERIFY[f'per-group norms @{tag}'] = (ok, f'worst {worst:.3f}% (want < 0.1%)')

    # the clip-then-sum tell-tale
    intruders = {k: v for k, v in got.items()
                 if k in ('E', 'P') and v > 100}
    VERIFY[f'clip_then_sum applied @{tag}'] = (
        not intruders,
        'E/P absent from top groups' if not intruders else f'INFLATED: {intruders}')

## V4. The remaining replay probes

These have no pre-recorded reference values, so the checks are *structural
and internal-consistency* rather than exact-match:

- **`row_attribution`** — do the per-row shares sum coherently, and is the
  concentration plausible? A top-1 share near 100% or near 1/n both deserve a
  second look.
- **`precision_cap`** — the arm at the run's **live** `precision_lr_max`
  must reproduce the recorded norm. That arm is a self-check; if it fails,
  the sweep means nothing.
- **`clip_order`** — `sum_then_clip` and `clip_then_sum` must differ, and
  the cosine must be ≤ 1. If they are identical the ablation is not actually
  varying anything.
- **`integrator`** — the arm at the run's live integrator (`baoab_cfc`) is
  again the self-check.

Each is slow (a full replay per arm), so run the ones you care about.

In [ ]:
from semsimula_diag.probes import row_attribution

res_rows = row_attribution.attribute_spike_rows(
    gpu_ctx, 87196, track=('V_theta.depth_code',), verbose=True)
t1 = res_rows.metrics['top1_share']
ok = 0.0 < t1 <= 1.0 and res_rows.metrics['n_rows'] > 0
VERIFY['row_attribution runs'] = (ok, f"top-1 share {t1:.1%} of "
                                      f"{res_rows.metrics['n_rows']} rows")

In [ ]:
from semsimula_diag.probes import precision_cap

# Include the LIVE budget as the self-check arm.
caps = precision_cap.replay_precision_cap_ablation(
    gpu_ctx, 87196, budgets=(PRECISION_LR_MAX, 0.25, None), verbose=True)

live_label = f'precision_lr_max={PRECISION_LR_MAX}'
live_arm = caps.get(live_label)
ok = live_arm is not None and (live_arm.fidelity_gap_pct or 1e9) < 0.01
VERIFY['precision_cap live-arm fidelity'] = (
    ok, f'{live_arm.fidelity_gap_pct:.4f}%' if live_arm else 'live arm missing')

In [ ]:
from semsimula_diag.probes import clip_order

res_clip = clip_order.replay_clip_ablation(
    gpu_ctx, 87196, groups=('E', 'P'), thresholds=(0.3, 0.1), verbose=True)

sts = res_clip.raw['sum_then_clip']; cts = res_clip.raw['clip_then_sum']
differs = any(abs(sts[g][t] - cts[g][t]) > 1e-6
              for g in sts for t in sts[g])
cos_ok = all(c <= 1.0 + 1e-5 for g in res_clip.raw['cosine_vs_sum_then_clip']
             for c in res_clip.raw['cosine_vs_sum_then_clip'][g].values())
VERIFY['clip_order arms differ'] = (
    differs and cos_ok,
    'orders produce different gradients' if differs else 'IDENTICAL - suspect')

In [ ]:
from semsimula_diag.probes import integrator

res_int = integrator.replay_integrator_ablation(
    gpu_ctx, 87196, integrators=('baoab_cfc', 'baoab_cfc_lowrank'),
    verbose=True)

live_arm = res_int.get('baoab_cfc')
ok = live_arm is not None and (live_arm.fidelity_gap_pct or 1e9) < 0.01
VERIFY['integrator live-arm fidelity'] = (
    ok, f'{live_arm.fidelity_gap_pct:.4f}%' if live_arm else 'missing')

## V5. Weight-only probes on GPU

These already passed on CPU; re-running here confirms nothing device-specific
broke, and gives you the `_best.pt` rank reading that section 9a could only
approximate from a spike checkpoint.

In [ ]:
from semsimula_diag.probes import stiffness, tau_saturation

# tau: pool-wide temperature at the capture
tau_res = tau_saturation.probe_gate_saturation(gpu_ctx, 87196, verbose=True)
VERIFY['tau_saturation'] = (
    tau_res.metrics['n_registers'] > 0,
    f"tau_min {tau_res.metrics['tau_min']:.3f} @ reg "
    f"{tau_res.metrics['tau_argmin']}")

# the rank decision, now on the BEST checkpoint (what SS2.1 actually specifies)
def _bp(n):
    import numpy as np
    _r = np.random.default_rng(0)
    xb, _ = get_batch(train_ids, min(BATCH_SIZE, n), BLOCK_SIZE, _r)
    return torch.from_numpy(xb)
gpu_ctx.batch_provider = _bp

specs = stiffness.spectrum_across_checkpoints(
    gpu_ctx, step_tags=(87196, 90360), n_batch=4)
print(f'\n{"checkpoint":34s} {"pr_p50":>8s} {"fro_p50":>9s}')
for label, r in specs.items():
    print(f'{label:34s} {r.metrics["pr_p50"]:8.3f} {r.metrics["fro_p50"]:9.5f}')
VERIFY['stiffness spectrum sweep'] = (len(specs) > 0, f'{len(specs)} checkpoints')

## V6. Summary

Everything in one table. **If any replay row fails, treat every replay-based
probe as unverified** and check `clip_cfg` first — a mismatched
clip-then-sum configuration is by far the most likely cause, and it fails
silently.

Please paste this table back so the README's status table can be updated.

In [ ]:
print(f'{"check":42s} {"result":>7s}   detail')
print('-' * 92)
n_pass = 0
for name, (ok, detail) in VERIFY.items():
    n_pass += bool(ok)
    print(f'{name:42s} {"PASS" if ok else "FAIL":>7s}   {detail}')
print('-' * 92)
print(f'{n_pass}/{len(VERIFY)} checks passed')
if n_pass != len(VERIFY):
    print('\nReplay probes remain UNVERIFIED. Check clip_cfg first:')
    print('  clip_then_sum_groups   =', sorted(gpu_cfg.clip_then_sum_groups))
    print('  clip_then_sum_threshold =', gpu_cfg.clip_then_sum_threshold)